In [1]:
# ========== 1. Install dependencies ==========
!pip install -q unsloth wandb datasets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 24.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 107.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.1 MB/s eta 0:00:00:00:01
  

In [ ]:
import os
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import wandb

# ========== 1. SECURITY: Load credentials from environment ==========
# Set these as environment variables, NOT in code!
# export HF_TOKEN="your_token"
# export WANDB_API_KEY="your_key"
HF_USERNAME = "TeslaInch"
HF_DATASET = "SCD-Instruction-Tuning"
HF_TOKEN = "your_api_key_here"
WANDB_API_KEY = "your_api_key_here"
WANDB_PROJECT = "scd-phi35-finetuning"

# Validate credentials exist
if not HF_TOKEN:
    raise ValueError("❌ HF_TOKEN not found. Set it: export HF_TOKEN='your_token'")
if not WANDB_API_KEY:
    raise ValueError("❌ WANDB_API_KEY not found. Set it: export WANDB_API_KEY='your_key'")

# ========== 2. Check CUDA availability ==========
if not torch.cuda.is_available():
    raise RuntimeError("❌ CUDA not available. GPU required for training.")
print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")

# ========== 3. Authenticate ==========
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
wandb.login(key=WANDB_API_KEY)

# ========== 4. Load dataset from Hugging Face ==========
try:
    dataset = load_dataset(
        f"{HF_USERNAME}/{HF_DATASET}",
        split="train",
        token=HF_TOKEN,
        trust_remote_code=False
    )
    print(f"✅ Loaded {len(dataset)} examples")
except Exception as e:
    raise RuntimeError(f"❌ Failed to load dataset: {e}")

# ========== 5. Validate dataset format ==========
required_fields = {"instruction", "input", "output"}
if not required_fields.issubset(set(dataset.column_names)):
    raise ValueError(
        f"❌ Dataset missing required fields. Found: {dataset.column_names}\n"
        f"Required: {required_fields}"
    )
print(f"✅ Dataset has correct schema: {dataset.column_names}")

# ========== 6. Load model with Unsloth ==========
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="microsoft/Phi-3.5-mini-instruct",
        max_seq_length=2048,
        load_in_4bit=True,
        device_map="auto",
    )
    print("✅ Model loaded successfully")
except Exception as e:
    raise RuntimeError(f"❌ Failed to load model: {e}")

# ========== 7. Apply LoRA adapters ==========
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=True,
)
print("✅ LoRA adapters applied")

# ========== 8. Format dataset for instruction tuning ==========
def format_example(example):
    """Format example with proper instruction template"""
    return {
        "text": (
            f"### Instruction:\n{example['instruction'].strip()}\n\n"
            f"### Input:\n{example['input'].strip()}\n\n"
            f"### Response:\n{example['output'].strip()}"
        )
    }

try:
    dataset = dataset.map(format_example, remove_columns=dataset.column_names)
    print(f"✅ Formatted {len(dataset)} examples")
except Exception as e:
    raise RuntimeError(f"❌ Failed to format dataset: {e}")

# ========== 9. Tokenize formatted dataset ==========
def tokenize_function(examples):
    """Tokenize the formatted text examples"""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=2048,
        padding="do_not_pad"
    )

try:
    dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    print(f"✅ Tokenized {len(dataset)} examples")
except Exception as e:
    raise RuntimeError(f"❌ Failed to tokenize dataset: {e}")

# ========== 10. Setup training arguments ==========
training_args = TrainingArguments(
    output_dir="./scd_phi35_adapter",
    num_train_epochs=6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    learning_rate=1e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,  # Keep only last 2 checkpoints
    optim="adamw_8bit",
    seed=42,
    report_to=["wandb"],
)

# ========== 11. Create trainer ==========
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

# ========== 12. Start training ==========
try:
    print("🚀 Starting training...")
    trainer.train()
    print("✅ Training completed successfully!")
except KeyboardInterrupt:
    print("⚠️  Training interrupted by user")
except Exception as e:
    print(f"❌ Training failed: {e}")
    raise

# ========== 13. Save model and tokenizer ==========
try:
    output_path = "./scd_phi35_adapter_final"
    model.save_pretrained(output_path)
    tokenizer.save_pretrained(output_path)
    print(f"✅ Model saved to {output_path}/")
except Exception as e:
    print(f"❌ Failed to save model: {e}")
    raise

# ========== 14. Cleanup ==========
wandb.finish()
print("✅ Training complete! Adapter ready for inference.")